## Import dependencies

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

## Position encoding

Standard sinusoidal absolute positional encoding of Classic Transformer:
$$
\begin{aligned}
PE(pos,2i)=\sin\left(pos/10000^{2i/d_{\text{model}}}\right) \\
PE(pos,2i+1)=\cos\left(pos/10000^{2i/d_{\text{model}}}\right) \\
\end{aligned}
$$

That is to say:
$$ X' = X + PE $$

In [ ]:
class PositionalEncoding(nn.Module):
    
    def __init__(self, d_model, max_len=5000):
        super(PositionalEncoding, self).__init__()
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        pe = pe.unsqueeze(0).transpose(0, 1)
        self.register_buffer('pe', pe)

    def forward(self, x):
        x = x + self.pe[:x.size(0), :]
        return x

## Multi-Head Self-attention

How to understand Atention Score especial this line code: 
```python
attn = F.softmax(scores, dim=-1)
```
Reference [QA1](./QA1_Encoder.ipynb.md)

In [ ]:
class MultiHeadSelfAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super(MultiHeadSelfAttention, self).__init__()
        self.num_heads = num_heads
        self.d_model = d_model
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        self.depth = d_model // num_heads

        self.wq = nn.Linear(d_model, d_model)
        self.wk = nn.Linear(d_model, d_model)
        self.wv = nn.Linear(d_model, d_model)
        self.dense = nn.Linear(d_model, d_model)

    def split_heads(self, x, batch_size):
        x = x.view(batch_size, -1, self.num_heads, self.depth)
        return x.transpose(1, 2)

    def forward(self, x):
        batch_size = x.size(0)
        q = self.split_heads(self.wq(x), batch_size)
        k = self.split_heads(self.wk(x), batch_size)
        v = self.split_heads(self.wv(x), batch_size)

        scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(self.depth)
        attn = F.softmax(scores, dim=-1) # attention scores
        context = torch.matmul(attn, v)
        context = context.transpose(1, 2).contiguous().view(batch_size, -1, self.d_model)
        return self.dense(context)

## Feed-Forward laye - MLP(Multi-Layer Perceptron)

Give an in-depth explanation of FFN. Why do we first upscale the dimension and then downscale it back to the original dimension? Reference: [QA2](./QA2_Encoder.ipynb.md)

In [ ]:
class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super(FeedForward, self).__init__()
        self.linear1 = nn.Linear(d_model, d_ff)
        self.linear2 = nn.Linear(d_ff, d_model)

    def forward(self, x):
        return self.linear2(F.relu(self.linear1(x)))

## Transformer encoder block

In [ ]:
class TransformerEncoderBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff, dropout=0.1):
        super(TransformerEncoderBlock, self).__init__()
        self.mha = MultiHeadSelfAttention(d_model, num_heads)
        self.ff = FeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        attn_output = self.mha(x)
        x = self.norm1(x + self.dropout(attn_output))
        ff_output = self.ff(x)
        x = self.norm2(x + self.dropout(ff_output))
        return x

## Encoder blocks

In [ ]:
class TransformerEncoder(nn.Module):
    def __init__(self, num_layers, d_model, num_heads, d_ff, dropout=0.1):
        super(TransformerEncoder, self).__init__()
        self.pos_encoding = PositionalEncoding(d_model)
        self.layers = nn.ModuleList([
            TransformerEncoderBlock(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
        self.norm = nn.LayerNorm(d_model)

    def forward(self, x):
        x = x + self.pos_encoding(x)
        for layer in self.layers:
            x = layer(x)
        return self.norm(x)

## Test the transformer encoder implementation

In [ ]:
batch_size = 2
seq_length = 5
d_model = 16
num_heads = 4
d_ff = 64
num_layers = 2

x = torch.rand(batch_size, seq_length, d_model)
encoder = TransformerEncoder(num_layers, d_model, num_heads, d_ff, dropout=0.1)
output = encoder(x)
print("Output shape:", output.shape)